In [2]:
import pandas as pd

# Just peek — don't load all 57M rows yet
df_peek = pd.read_csv(
    "data/rating_complete.csv",
    nrows=5,
    dtype={"user_id": "int32", "anime_id": "int32", "rating": "int8"},
)
print("Columns:", df_peek.columns.tolist())
print("Sample rows:")
print(df_peek)

Columns: ['user_id', 'anime_id', 'rating']
Sample rows:
   user_id  anime_id  rating
0        0       430       9
1        0      1004       5
2        0      3010       7
3        0       570       7
4        0      2762       9


In [3]:
import pandas as pd

print("Loading rating_complete.csv... (may take 30-60s)")
ratings = pd.read_csv(
    "data/rating_complete.csv",
    dtype={"user_id": "int32", "anime_id": "int32", "rating": "int8"},
)
print(f"Shape:        {ratings.shape}")
print(f"RAM usage:    {ratings.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Unique users: {ratings['user_id'].nunique():,}")
print(f"Unique anime: {ratings['anime_id'].nunique():,}")
print(f"Rating range: {ratings['rating'].min()} – {ratings['rating'].max()}")
print("\nRating distribution:")
print(ratings["rating"].value_counts().sort_index())

Loading rating_complete.csv... (may take 30-60s)
Shape:        (57633278, 3)
RAM usage:    0.52 GB
Unique users: 310,059
Unique anime: 16,872
Rating range: 1 – 10

Rating distribution:
rating
1       333419
2       405556
3       696048
4      1455102
5      3436250
6      6849293
7     13325549
8     14642156
9      9773857
10     6716048
Name: count, dtype: int64


In [4]:
# Count ratings per user, filter those with 100+
user_counts = ratings.groupby("user_id").size().reset_index(name="n_ratings")
power_users = user_counts[user_counts["n_ratings"] >= 100].sort_values(
    "n_ratings", ascending=False
)

print(f"Users with 100+ ratings: {len(power_users):,}")
print(f"\nTop 20 most active users:")
print(power_users.head(20).to_string(index=False))

# Save for use in next chunks
power_user_ids = set(power_users["user_id"].tolist())

Users with 100+ ratings: 167,420

Top 20 most active users:
 user_id  n_ratings
  189037      15455
  162615      14864
   68042      13462
  283786      12778
  259790       9996
  291207       9678
  182280       9607
  277841       8724
  336459       8567
   10255       8215
  312302       8006
  328195       7779
  300428       7503
  276953       7277
  190748       7032
  345498       6872
   64807       6656
  122341       6485
   85106       6464
  299795       6152


In [6]:
import pandas as pd

anime_meta = pd.read_parquet("processed/anime_meta.parquet")
print("Columns:", anime_meta.columns.tolist())

target_anime = {
    "Action/Shounen": ["Naruto", "Dragon Ball Z", "Bleach", "One Piece", "Fairy Tail"],
    "Romance/Drama": ["Clannad", "Toradora!", "Your Lie in April", "Anohana", "Nana"],
    "Psychological": ["Death Note", "Code Geass", "Monster", "Steins;Gate", "Akira"],
    "Sci-Fi/Mecha": [
        "Neon Genesis Evangelion",
        "Gurren Lagann",
        "Cowboy Bebop",
        "Ghost in the Shell",
    ],
    "Mixed/General": [
        "Fullmetal Alchemist: Brotherhood",
        "Hunter x Hunter (2011)",
        "Attack on Titan",
    ],
}

# ---- fixed: anime_id instead of MAL_ID ----
for profile, titles in target_anime.items():
    found = anime_meta[anime_meta["Name"].isin(titles)][["anime_id", "Name", "Genres"]]
    print(f"\n--- {profile} ---")
    if found.empty:
        print("  !! No exact matches — printing fuzzy candidates:")
        # fuzzy fallback: partial match on any title in the list
        pattern = "|".join(titles)
        fuzzy = anime_meta[
            anime_meta["Name"].str.contains(pattern, case=False, na=False)
        ][["anime_id", "Name", "Genres"]]
        print(fuzzy.to_string(index=False))
    else:
        print(found.to_string(index=False))

Columns: ['anime_id', 'Name', 'Score', 'Genres', 'English name', 'Type', 'Episodes', 'Studios', 'Source', 'Rating', 'Members', 'Favorites', 'Completed', 'Ranked', 'Popularity', 'display_name', 'genre_list']

--- Action/Shounen ---
 anime_id          Name                                                                 Genres
       20        Naruto          Action, Adventure, Comedy, Super Power, Martial Arts, Shounen
      269        Bleach          Action, Adventure, Comedy, Super Power, Supernatural, Shounen
      813 Dragon Ball Z Action, Adventure, Comedy, Fantasy, Martial Arts, Shounen, Super Power
     6702    Fairy Tail                     Action, Adventure, Comedy, Magic, Fantasy, Shounen

--- Romance/Drama ---
 anime_id      Name                                                      Genres
      877      Nana        Music, Slice of Life, Comedy, Drama, Romance, Shoujo
     2167   Clannad Comedy, Drama, Romance, School, Slice of Life, Supernatural
     4224 Toradora!            

In [9]:
import pandas as pd
import pickle
import numpy as np

# ── Load artifacts ──────────────────────────────────────────────────────────
with open("processed/mappings.pkl", "rb") as f:
    mappings = pickle.load(f)

valid_user_ids = set(mappings["user2idx"].keys())
anime_meta = pd.read_parquet("processed/anime_meta.parquet")

print(f"Valid users in model : {len(valid_user_ids):,}")
print(f"Anime in metadata    : {len(anime_meta):,}")

# ── Persona definitions — genre keyword → label ─────────────────────────────
personas = {
    "Action/Shounen": ["Action", "Shounen", "Martial Arts", "Super Power"],
    "Romance/Drama": ["Romance", "Drama"],
    "Psychological": ["Psychological", "Thriller", "Mystery"],
    "Sci-Fi/Mecha": ["Sci-Fi", "Mecha", "Space"],
    "Fantasy/Adventure": ["Fantasy", "Adventure", "Magic"],
    "Horror/Dark": ["Horror", "Demons", "Supernatural"],
    "Slice of Life": ["Slice of Life", "School", "Everyday"],
    "Comedy": ["Comedy", "Parody"],
    "Sports": ["Sports"],
    "Ecchi/Harem": ["Ecchi", "Harem"],
}


# ── Helper: get anime_ids matching ANY of the genre keywords ────────────────
def genre_anime_ids(keywords):
    pattern = "|".join(keywords)
    return set(
        anime_meta[anime_meta["Genres"].str.contains(pattern, case=False, na=False)][
            "anime_id"
        ].tolist()
    )


# ── Pre-filter ratings to valid model users only (do once, speeds up loops) ─
print("\nFiltering ratings to valid model users...")
ratings_valid = ratings[ratings["user_id"].isin(valid_user_ids)].copy()
print(f"Ratings after filter : {len(ratings_valid):,}")

user_counts = ratings_valid.groupby("user_id").size().reset_index(name="n_ratings")


# ── Single-persona finder ───────────────────────────────────────────────────
def find_persona_users(keywords, min_ratings=100, top_n=8):
    ids = genre_anime_ids(keywords)
    subset = ratings_valid[
        ratings_valid["anime_id"].isin(ids) & (ratings_valid["rating"] >= 8)
    ]
    top = (
        subset.groupby("user_id")
        .size()
        .reset_index(name="genre_hits")
        .sort_values("genre_hits", ascending=False)
        .head(top_n * 3)  # oversample then filter
    )
    top = top.merge(user_counts, on="user_id")
    top = top[top["n_ratings"] >= min_ratings].head(top_n)
    top["hit_rate"] = (top["genre_hits"] / top["n_ratings"] * 100).round(1)
    return top[["user_id", "n_ratings", "genre_hits", "hit_rate"]]


# ── Mixed-taste finder — user must have hits across N distinct genres ────────
def find_mixed_users(min_genres=5, min_hits_per_genre=5, min_ratings=150, top_n=8):
    """
    Find users who rated highly (>=8) across at least `min_genres` different
    persona categories — true generalists, not dominated by one genre.
    """
    user_genre_hits = {}  # user_id → {persona: count}

    for persona_name, keywords in personas.items():
        ids = genre_anime_ids(keywords)
        subset = ratings_valid[
            ratings_valid["anime_id"].isin(ids) & (ratings_valid["rating"] >= 8)
        ]
        grouped = subset.groupby("user_id").size()
        for uid, cnt in grouped.items():
            if cnt >= min_hits_per_genre:
                user_genre_hits.setdefault(uid, {})[persona_name] = cnt

    rows = []
    for uid, genre_dict in user_genre_hits.items():
        if len(genre_dict) >= min_genres:
            rows.append(
                {
                    "user_id": uid,
                    "genres_covered": len(genre_dict),
                    "genres_list": ", ".join(sorted(genre_dict.keys())),
                }
            )

    if not rows:
        print("No mixed users found — try lowering min_genres or min_hits_per_genre")
        return pd.DataFrame()

    result = (
        pd.DataFrame(rows)
        .sort_values("genres_covered", ascending=False)
        .merge(user_counts, on="user_id")
        .query(f"n_ratings >= {min_ratings}")
        .head(top_n)
    )
    return result[["user_id", "n_ratings", "genres_covered", "genres_list"]]


# ── Run all personas ────────────────────────────────────────────────────────
persona_candidates = {}

print("\n" + "=" * 65)
for name, keywords in personas.items():
    result = find_persona_users(keywords)
    persona_candidates[name] = result
    print(f"\n{'='*65}")
    print(f"  PERSONA : {name}")
    print(f"  Keywords: {', '.join(keywords)}")
    print(f"{'='*65}")
    print(
        result.to_string(index=False)
        if not result.empty
        else "  !! No candidates found"
    )

# ── Run mixed persona ───────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  PERSONA : Mixed/Balanced (generalist across all genres)")
print(f"{'='*65}")
mixed = find_mixed_users(min_genres=5, min_hits_per_genre=5, min_ratings=150)
persona_candidates["Mixed/Balanced"] = mixed
print(mixed.to_string(index=False) if not mixed.empty else "  !! No candidates found")

# ── Summary table — one best candidate per persona ─────────────────────────
print(f"\n{'='*65}")
print("  SUMMARY — Top candidate per persona")
print(f"{'='*65}")
for name, df in persona_candidates.items():
    if not df.empty:
        row = df.iloc[0]
        uid = int(row["user_id"])
        nr = int(row["n_ratings"])
        print(f"  {name:<22} → user_id={uid:<8}  n_ratings={nr}")
    else:
        print(f"  {name:<22} → NO CANDIDATE FOUND")

Valid users in model : 223,565
Anime in metadata    : 10,502

Filtering ratings to valid model users...
Ratings after filter : 55,857,448


  PERSONA : Action/Shounen
  Keywords: Action, Shounen, Martial Arts, Super Power
 user_id  n_ratings  genre_hits  hit_rate
  162615      14864        3700      24.9
  127483       5732        2201      38.4
  312302       8006        2117      26.4
  117521       3062        1573      51.4
  298783       4598        1525      33.2
  327150       3462        1518      43.8
  332300       4147        1496      36.1
  168939       3202        1444      45.1

  PERSONA : Romance/Drama
  Keywords: Romance, Drama
 user_id  n_ratings  genre_hits  hit_rate
  162615      14864        2671      18.0
  312302       8006        1622      20.3
  127483       5732        1564      27.3
  327150       3462        1384      40.0
  298783       4598        1215      26.4
  332300       4147        1137      27.4
  168939       3202        1055      32.9
  117521  